# 02 · Silver: clean and standardize

Turns the raw bronze copy into reliable tables, one per entity:
- Correct data types (timestamps, `DECIMAL` for money, integers).
- One row per key, deduplicated with `QUALIFY ROW_NUMBER()` (latest load wins).
- Normalized text (trim, case, accents) and standardized keys (zip codes padded to 5 digits).
- Explicit cleaning rules: no `not_defined` payments, installments >= 1, review scores 1-5, prices >= 0.

Data quality checks for this layer run in `05_quality_checks`.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.silver;

-- Reusable rule to normalize city names (lowercase, no spaces at the ends, no accents)
CREATE OR REPLACE FUNCTION workspace.silver.normalize_city(s STRING)
RETURNS STRING
RETURN translate(lower(trim(s)), 'áàâãäéèêëíìîïóòôõöúùûüç', 'aaaaaeeeeiiiiooooouuuuc');

In [0]:
%sql
-- Orders: typed timestamps and delivery delay vs the promised date
CREATE OR REPLACE TABLE workspace.silver.orders AS
SELECT order_id, customer_id,
       lower(trim(order_status))                     AS order_status,
       to_timestamp(order_purchase_timestamp)        AS purchase_ts,
       to_timestamp(order_approved_at)               AS approved_ts,
       to_timestamp(order_delivered_customer_date)   AS delivered_ts,
       to_timestamp(order_estimated_delivery_date)   AS estimated_delivery_ts,
       datediff(to_timestamp(order_delivered_customer_date),
                to_timestamp(order_estimated_delivery_date)) AS delivery_delay_days
FROM workspace.bronze.orders
WHERE order_id IS NOT NULL
QUALIFY row_number() OVER (PARTITION BY order_id ORDER BY _ingest_ts DESC) = 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Customers: customer_id changes with every order; customer_unique_id is the real customer
CREATE OR REPLACE TABLE workspace.silver.customers AS
SELECT customer_id,
       customer_unique_id,
       lpad(trim(customer_zip_code_prefix), 5, '0')      AS zip_prefix,
       workspace.silver.normalize_city(customer_city)    AS city,
       upper(trim(customer_state))                       AS state
FROM workspace.bronze.customers
WHERE customer_id IS NOT NULL
QUALIFY row_number() OVER (PARTITION BY customer_id ORDER BY _ingest_ts DESC) = 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.sellers AS
SELECT seller_id,
       lpad(trim(seller_zip_code_prefix), 5, '0')     AS zip_prefix,
       workspace.silver.normalize_city(seller_city)   AS city,
       upper(trim(seller_state))                      AS state
FROM workspace.bronze.sellers
WHERE seller_id IS NOT NULL
QUALIFY row_number() OVER (PARTITION BY seller_id ORDER BY _ingest_ts DESC) = 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.order_items AS
SELECT order_id,
       try_cast(order_item_id AS INT)            AS order_item_id,
       product_id, seller_id,
       to_timestamp(shipping_limit_date)         AS shipping_limit_ts,
       try_cast(price AS DECIMAL(12,2))          AS price,
       try_cast(freight_value AS DECIMAL(12,2))  AS freight_value
FROM workspace.bronze.order_items
WHERE order_id IS NOT NULL
  AND try_cast(price AS DECIMAL(12,2)) >= 0
QUALIFY row_number() OVER (PARTITION BY order_id, order_item_id ORDER BY _ingest_ts DESC) = 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Payments: drop 'not_defined' payments and force at least 1 installment
CREATE OR REPLACE TABLE workspace.silver.order_payments AS
SELECT order_id,
       try_cast(payment_sequential AS INT)                  AS payment_sequential,
       lower(trim(payment_type))                            AS payment_type,
       greatest(try_cast(payment_installments AS INT), 1)   AS installments,
       try_cast(payment_value AS DECIMAL(12,2))             AS payment_value
FROM workspace.bronze.order_payments
WHERE order_id IS NOT NULL
  AND lower(trim(payment_type)) <> 'not_defined'
QUALIFY row_number() OVER (PARTITION BY order_id, payment_sequential ORDER BY _ingest_ts DESC) = 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Reviews: valid scores only, one-line comments, one review per order (latest answer)
CREATE OR REPLACE TABLE workspace.silver.order_reviews AS
SELECT review_id, order_id,
       try_cast(review_score AS INT)                         AS review_score,
       nullif(trim(review_comment_title), '')                AS comment_title,
       nullif(trim(regexp_replace(review_comment_message, '[\\r\\n]+', ' ')), '') AS comment_message,
       to_timestamp(review_creation_date)                    AS created_ts,
       to_timestamp(review_answer_timestamp)                 AS answered_ts
FROM workspace.bronze.order_reviews
WHERE order_id IS NOT NULL
  AND try_cast(review_score AS INT) BETWEEN 1 AND 5
QUALIFY row_number() OVER (PARTITION BY order_id
                           ORDER BY to_timestamp(review_answer_timestamp) DESC) = 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Products: fix column typos and translate categories to English
CREATE OR REPLACE TABLE workspace.silver.products AS
SELECT p.product_id,
       coalesce(t.product_category_name_english, p.product_category_name, 'unknown') AS category,
       try_cast(p.product_name_lenght AS INT)         AS name_length,
       try_cast(p.product_description_lenght AS INT)  AS description_length,
       try_cast(p.product_photos_qty AS INT)          AS photos_qty,
       try_cast(p.product_weight_g AS INT)            AS weight_g,
       try_cast(p.product_length_cm AS INT)           AS length_cm,
       try_cast(p.product_height_cm AS INT)           AS height_cm,
       try_cast(p.product_width_cm AS INT)            AS width_cm
FROM workspace.bronze.products p
LEFT JOIN workspace.bronze.category_translation t
       ON trim(p.product_category_name) = trim(t.product_category_name)
WHERE p.product_id IS NOT NULL
QUALIFY row_number() OVER (PARTITION BY p.product_id ORDER BY p._ingest_ts DESC) = 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Geolocation: from ~1M rows to one row per zip prefix, coordinates inside Brazil only
CREATE OR REPLACE TABLE workspace.silver.geolocation AS
SELECT lpad(trim(geolocation_zip_code_prefix), 5, '0')                  AS zip_prefix,
       avg(try_cast(geolocation_lat AS DOUBLE))                         AS lat,
       avg(try_cast(geolocation_lng AS DOUBLE))                         AS lng,
       mode(workspace.silver.normalize_city(geolocation_city))          AS city,
       mode(upper(trim(geolocation_state)))                             AS state
FROM workspace.bronze.geolocation
WHERE try_cast(geolocation_lat AS DOUBLE) BETWEEN -34 AND 6
  AND try_cast(geolocation_lng AS DOUBLE) BETWEEN -74 AND -34
GROUP BY 1;

num_affected_rows,num_inserted_rows
